
# 2. Auxiliaries: UART1 & SPI1, SPI2 (p8~p22)
# 2.1 Overview  
- offset 是 0x7E215000-0x7E000000=0x00215000
- 對應到 aux.h 的 struct

|Addresss| Register Name   |Description         | Size|
|----------|---------------|--------------------|-|
|0x7E215000|AUX_IRQ        |Aux interrupt status|3|
|0x7E215004|AUX_enables    |AUX enable          |3|
|0x7E215040|AUX_MU_io      |mini uart IO data   |8|
|0x7E215044|AUX_MU_ier |mini uart interrupt enable|8|
|0x7E215048|AUX_MU_iir |mini uart interrupt id    |8|
|0x7E21504C|AUX_MU_lcr     |mu line control       |8|
|0x7E215050|AUX_MU_mcr     |mu modem control      |8|
|0x7E215054|AUX_MU_lsr     |mu line status        |8|
|0x7E215058|AUX_MU_msr     |mu modem status       |8|
|0x7E21505C|AUX_MU_scratch |mu scratch            |8|
|0x7E215060|AUX_MU_cntrl   |mu extra control      |8|
|0x7E215064|AUX_MU_stat    |mu extra status       |32|
|0x7E215068|AUX_MU_baud    |mu baud rate          |8|

- 0x7E215068~0x7E2150C4 is SPI0 and SPI1 

- 每個 addr 對應一個 byte, 所以幾乎都是跳 4 個 byte, = 32bits

## p11 baud rate :  
  $$
  \text{baudrate} = \frac{\text{system\_clock\_freq}}{8\times(\text{baudrate\_reg+1})}
  $$
  - sys_clock_fre = 250 M Hz
  - 設定 mu_baud_rate_reg = 270, 則
  $$
   baudrate=   \frac{\text{250000000}}{8\times(\text{270+1})} \approx 115314 \approx 115200
  $$
## 2.2.2 mini UART register detail

### p11 AUX_MU_IO_REG (0x7E21 5040)
- used to write data to and read data from the UART FIFOs 
- 8 bit: 可以寫 char
- 用在 mini_uart.c: uart_send
- 3 種 mode 取決於 AUX_MU_LCR bit 7:DLAB
  - 我們 AUX_MU_LCR 設成 3, 所以 DLAB = 0


|Bits|FieldName |Description                        |Type|Reset|
|----|----------|-----------------------------------|----|-----|
|31:8|          |Reserved                           |    |     |
|7:0 |LS 8 bits |If AUX_MU_lcr DLAB=1, assess to    |R/W |0    |
|    |          |baud rate reg                      |    |     |
|7:0 |Tx data   |If DLAB=0, Data written is put in  |W   |0    |
|    |          |the transmit FIFO                  |    |     |   
|7:0 |Rx data   |If DLAB=0, Data read is taken      |R   |0    | 
|    |          |from Rx FIFO                       |    |     |   


### p12-14 datasheet 其實寫錯了, [datasheet error](https://elinux.org/BCM2835_datasheet_errata#p12)

### p12 AUX_MU_IER_REG. 
  - **Title should be AUX_MU_IER_REG.** 
  - The name in Synopsis is correct
  - These are R/W bits not read only
  - Bits 3:2 are marked as don't care, but are actually required in order to receive interrupts.
  - Bits 1:0 are swaped. bit 0 is receive interrupt and bit 1 is transmit. 
### p13: AUX_MU_IIR_REG. 
  - The title should be AUX_MU_IIR_REG. 
  - The name in the Synopsis is correct 
### p14: AUX_MU_lcr 
  - 寫錯了 AUX_MU_lcr: 5:2 reserve 0, 
  - [1:0]-> 00->:bit mode , 11:8bit mode 所以要設定成 3才是 8bit mode
  - **DLAB access** 很重要, 會決定 AUX_MU_IO_REG 的 mode

|Bits|FieldName  |Description                        |Type|Reset|
|----|-----------|-----------------------------------|----|-----|
|31:8|           |Reserved                           |    |     |
|7   |DLAB access|if set first **two** mini uart reg |R/W |0    |
|    |           |give access the baudrate reg       |    |     |
|6   |Break      |if set UART1_TX is pull low        |R/W |0    |
|5:2 |           |Reserved                           |    |0    | 
|1:0 |data size  |if 00 UART work in 7bit mode       |R/W |0    | 
|    |           |if 11 UART work in 8bit mode       |    |     | 



### p14 AUX_MU_MCR 
- 沒用到

### p15 AUX_MU_LSR   
- 在 miniuart.c
- uart_send 只有在 bit 5=1, Tx empty 的時候才會 send
  - 0x20 = 0010 0000 = bit 5
- uard_recv 只有在 bit 0=1, Rx fifo 不為空的時候才會 read    

|Bits|FieldName |Description                        |Type|Reset|
|----|----------|-----------------------------------|----|-----|
|31:8|          |Reserved                           |    |     |
|7   |          |Reserved                           |    |0    | 
|6   |Tx idle   |set if Tx FIFO empty and Tx idle   |R   |1    |
|5   |Tx empty  |set if Tx FIFO can accept >= 1 byte|R   |1    |
|4:2 |          |Reserved                           |    |0    | 
|1   |Rx Overrun|Rx is full and receive too many    |    |     |
|0   |Data ready|set if Rx FIFO holds >= 1 symbol   |    |     | 

### p16: AUX_MU_cntrl[7:0]-> 1:Tx enable, 0:Rx enable



# 2.2 Mini Uart   (p10)
# 2.3 SPI   

